> 在线讲义：[第 56 章 · VIO：IMU 运动学、偏置与噪声、流形上的预积分、可观测性](https://wikieden.github.io/math-and-eval-stats/lessons/56-vio.html) · 本书《数学与实验统计》

# 第 56 章 · VIO：IMU 运动学、偏置与噪声、流形上的预积分、可观测性

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/56-vio.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 漂移 | 解析式 | `drift_from_bias(t)`、`angle_random_walk(t)` | 0.10 rad、2.5 m、16.4 m |
| 第三步 预积分 | 式 (33) | `preintegrate(omega, acc)` | ΔR 29.22° |
| 第三步 组合 | p_j 公式 | `integrate_world` vs 预积分 | 完全一致 |
| 第四步 偏置修正 | 一阶 | `bias_update_error(dbg, omega)` | 9.9e-3 → 4.2e-6 |

例子：200 Hz 的 IMU、1 秒内恒定角速度和比力；预积分量 ΔR、Δv、Δp 不依赖起点的速度和姿态；
陀螺 / 加速度计偏置造成的误差随时间的增长（解析）；角度随机游走；偏置变化时的一阶修正。
里程碑 ME-M34 的“IMU 积分漂移模拟”留给自己做；这里漂移只用公式算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
# 下面几段来自 ch19_lie_groups.py（第 19 章）
import numpy as np
def hat(w: np.ndarray) -> np.ndarray:
    """向量 → 反对称矩阵 [w]×：[w]× v = w × v。"""
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def vee(m: np.ndarray) -> np.ndarray:
    """反对称矩阵 → 向量。"""
    return np.array([m[2, 1], m[0, 2], m[1, 0]])

def so3_exp(phi: np.ndarray) -> np.ndarray:
    """切空间 → SO(3)：Rodrigues 公式 R = I + sinθ/θ·[φ]× + (1 − cosθ)/θ²·[φ]×²。"""
    th = np.linalg.norm(phi)
    k = hat(phi)
    if th < 1e-9:
        return np.eye(3) + k
    return np.eye(3) + np.sin(th) / th * k + (1 - np.cos(th)) / th**2 * k @ k

def so3_log(r: np.ndarray) -> np.ndarray:
    """SO(3) → 切空间：θ = arccos((tr R − 1)/2)，φ = θ/(2 sinθ)·vee(R − Rᵀ)（θ 不接近 0、π 时）。"""
    th = np.arccos(np.clip((np.trace(r) - 1) / 2, -1, 1))
    if th < 1e-9:
        return vee(r - r.T) / 2
    return th / (2 * np.sin(th)) * vee(r - r.T)

def right_jacobian(phi: np.ndarray) -> np.ndarray:
    """SO(3) 的右雅可比 J_r(φ)：exp(φ + δ) ≈ exp(φ)·exp(J_r δ)。"""
    th = np.linalg.norm(phi)
    k = hat(phi)
    if th < 1e-9:
        return np.eye(3) - 0.5 * k
    return np.eye(3) - (1 - np.cos(th)) / th**2 * k + (th - np.sin(th)) / th**3 * k @ k

G = np.array([0.0, 0.0, -9.81])
DT = 1 / 200

**函数 `preintegrate`**：在 i 时刻的机体系里累积：ΔR ← ΔR·Exp((ω − b_g)dt)，Δv ← Δv + ΔR a dt，Δp ← Δp + Δv dt + ½ΔR a dt²（Forster 2015 式 (33)）。

In [ ]:
def preintegrate(omega: np.ndarray, acc: np.ndarray, n: int = 200, bg: np.ndarray | None = None):
    """在 i 时刻的机体系里累积：ΔR ← ΔR·Exp((ω − b_g)dt)，Δv ← Δv + ΔR a dt，Δp ← Δp + Δv dt + ½ΔR a dt²（Forster 2015 式 (33)）。"""
    bg = np.zeros(3) if bg is None else bg
    dr, dv, dp = np.eye(3), np.zeros(3), np.zeros(3)
    for _ in range(n):
        dp = dp + dv * DT + 0.5 * dr @ acc * DT * DT
        dv = dv + dr @ acc * DT
        dr = dr @ so3_exp((omega - bg) * DT)
    return dr, dv, dp

**函数 `integrate_world`**：直接在世界系里逐步积分：a_world = R a + g。

In [ ]:
def integrate_world(r0: np.ndarray, v0: np.ndarray, p0: np.ndarray, omega: np.ndarray, acc: np.ndarray, n: int = 200):
    """直接在世界系里逐步积分：a_world = R a + g。"""
    r, v, p = r0.copy(), v0.copy(), p0.copy()
    for _ in range(n):
        aw = r @ acc + G
        p = p + v * DT + 0.5 * aw * DT * DT
        v = v + aw * DT
        r = r @ so3_exp(omega * DT)
    return r, v, p

**函数 `drift_from_bias`**：恒定偏置的影响：陀螺偏置 → 角度误差 b_g·t；加速度计偏置 → 位置误差 ½b_a t²；

In [ ]:
def drift_from_bias(t: float, bg: float = 0.01, ba: float = 0.05) -> tuple[float, float, float]:
    """恒定偏置的影响：陀螺偏置 → 角度误差 b_g·t；加速度计偏置 → 位置误差 ½b_a t²；
    陀螺偏置造成的倾斜让重力漏进水平方向：位置误差约 (1/6)·g·b_g·t³。"""
    return bg * t, 0.5 * ba * t * t, 9.81 * bg * t ** 3 / 6

**函数 `angle_random_walk`**：白噪声积分：角度标准差 σ_g·√t（σ_g 单位 rad/s/√Hz）。

In [ ]:
def angle_random_walk(t: float, sigma: float = 0.005) -> float:
    """白噪声积分：角度标准差 σ_g·√t（σ_g 单位 rad/s/√Hz）。"""
    return sigma * np.sqrt(t)

**函数 `bias_update_error`**：偏置改了 δb_g：重新积分 vs 一阶修正 ΔR(b + δb) ≈ ΔR(b)·Exp(J δb)，J = −Σ ΔR_{k+1→j}ᵀ J_r(ω_k dt) dt。返回 (不修正的误差, 一阶修正后的误差)（弧度）。

In [ ]:
def bias_update_error(dbg: np.ndarray, omega: np.ndarray) -> tuple[float, float]:
    """偏置改了 δb_g：重新积分 vs 一阶修正 ΔR(b + δb) ≈ ΔR(b)·Exp(J δb)，J = −Σ ΔR_{k+1→j}ᵀ J_r(ω_k dt) dt。返回 (不修正的误差, 一阶修正后的误差)（弧度）。"""
    dr0, _, _ = preintegrate(omega, np.zeros(3))
    dr_true, _, _ = preintegrate(omega, np.zeros(3), bg=dbg)
    jac = np.zeros((3, 3))
    rs = [np.eye(3)]
    for _ in range(200):
        rs.append(rs[-1] @ so3_exp(omega * DT))
    for k in range(200):
        r_kj = rs[k + 1].T @ rs[200]
        jac -= r_kj.T @ right_jacobian(omega * DT) * DT
    approx = dr0 @ so3_exp(jac @ dbg)
    return float(np.linalg.norm(so3_log(dr0.T @ dr_true))), float(np.linalg.norm(so3_log(approx.T @ dr_true)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
omega, acc = np.array([0.0, 0.1, 0.5]), np.array([0.5, 0.0, 9.81])

In [ ]:
print("\n第一步 预积分：Δ 量只依赖这段时间的 IMU 读数")
dr, dv, dp = preintegrate(omega, acc)
print(f"  1 秒、ω = {omega.tolist()}、a = {acc.tolist()}：ΔR 的转角 {np.degrees(np.linalg.norm(so3_log(dr))):.2f}°，Δv = {np.round(dv, 4).tolist()}，Δp = {np.round(dp, 4).tolist()}")
for v0 in (np.zeros(3), np.array([2.0, -1.0, 0.5])):
    r0 = so3_exp(np.array([0.1, 0.0, 0.3]))
    rj, vj, pj = integrate_world(r0, v0, np.zeros(3), omega, acc)
    pred = v0 * 1.0 + 0.5 * G + r0 @ dp
    print(f"  起始速度 {v0.tolist()}：直接积分 p_j = {np.round(pj, 4).tolist()}；用预积分拼 = {np.round(pred, 4).tolist()}")

In [ ]:
print("\n第二步 偏置：误差按 t、t²、t³ 增长")
for t in (1, 10, 60):
    a, b, c = drift_from_bias(t)
    print(f"  t = {t} s：陀螺偏置 0.01 rad/s → 角度 {a:.2f} rad；加速度计偏置 0.05 → 位置 {b:.2f} m；倾斜漏重力 → 位置 {c:.1f} m")

In [ ]:
print("\n第三步 噪声：随机游走按 √t 增长")
for t in (1, 100, 3600):
    print(f"  t = {t} s：角度随机游走标准差 {angle_random_walk(t):.3f} rad")

In [ ]:
print("\n第四步 偏置变了，不用重新积分：一阶修正")
for db in (0.001, 0.01):
    e0, e1 = bias_update_error(np.array([db, 0, 0]), omega)
    print(f"  陀螺偏置改 {db} rad/s：不修正误差 {e0:.2e} rad，一阶修正后 {e1:.2e} rad")

## 3. 代码里才看得到的

1. **更新顺序**：先用旧的 Δv、ΔR 更新 Δp，再更新 Δv，最后更新 ΔR；顺序错了会和世界系积分对不上（误差 O(dt)）。
2. **组合公式里重力单独加**：预积分的 a 是原始比力（含重力反作用 +9.81），重力项 ½gΔt² 在组合时才加上。
3. **漂移只用解析式算**：没有模拟带噪声和偏置的 IMU 积分漂移；模拟并画出来留给自己做（里程碑 ME-M34）。

## 4. 自己改一改

1. 把 `preintegrate` 的采样率改成 100 Hz，Δp 变化多少？
2. 给 `bias_update_error` 加上加速度计偏置的修正（∂Δv/∂b_a = −Σ ΔR_ik dt），看效果。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 角速度积一次成姿态，比力转世界加重力积两次成位置；
> 偏置误差 t、t²、t³，噪声根号 t，歪一点重力就漏进来；
> 预积分在 i 系里累，和起点无关，改起点不重积；
> 偏置变了一阶修正；位置和偏航永远不可观。